# KrishiYield: Corn Hyperparameter Tuning & Model Finalization
**Project:** AI-Powered Corn and Soybean Yield Prediction System  
**Selected Champion Architecture:** Extreme Gradient Boosting (XGBoost Regressor)  
**Objective:** Fine-tune tree depth, shrinkage (learning rate), subsampling, and feature subsampling via 3-Fold Grid Search.

---
### Tuning Rationale
While default XGBoost achieved $R^2 = 0.9328$, systematic grid search optimizes tree regularization to mitigate overfitting to localized county weather patterns and enhance generalization across unseen climate shocks.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import matplotlib.pyplot as plt

MODEL_PATH = r"../backend/models/corn_model_final.pkl"
artifact = joblib.load(MODEL_PATH)
print("Loaded production model artifact successfully.")
print(f"Model Name: {artifact['model_name']}")
print(f"Optimal Hyperparameters: {artifact['best_params']}")

Loaded production model artifact successfully.
Model Name: Tuned XGBoost Regressor
Optimal Hyperparameters: {'colsample_bytree': 0.85, 'learning_rate': 0.07, 'max_depth': 5, 'n_estimators': 300, 'subsample': 0.85}


## 1. Hyperparameter Search Space & Grid Results
The grid search evaluated:
- `max_depth`: [5, 6] (controlling tree complexity and interaction depth)
- `learning_rate`: [0.04, 0.07] (controlling shrinkage and convergence rate)
- `n_estimators`: [200, 300] (number of boosting rounds)
- `subsample`: [0.85] (stochastic row subsampling for variance reduction)
- `colsample_bytree`: [0.85] (feature subsampling per tree)

In [2]:
metrics = artifact['metrics']
print("Final Test Set Evaluation Metrics (Untouched 800 Observations):")
print(f"  R² Score:  {metrics['test_r2']:.4f}")
print(f"  RMSE:      {metrics['test_rmse']:.2f} bu/acre ({metrics['test_rmse'] * 0.06277:.2f} metric tons/ha)")
print(f"  MAE:       {metrics['test_mae']:.2f} bu/acre ({metrics['test_mae'] * 0.06277:.2f} metric tons/ha)")
print(f"  MAPE:      {metrics['test_mape']:.2f}% (Average percentage error < 6%)")

Final Test Set Evaluation Metrics (Untouched 800 Observations):
  R² Score:  0.9333
  RMSE:      9.69 bu/acre (0.61 metric tons/ha)
  MAE:       7.15 bu/acre (0.45 metric tons/ha)
  MAPE:      5.76% (Average percentage error < 6%)


## 2. Feature Importance Analysis
We extract Gini / gain-based feature importances from the tuned model to understand the driving agronomic factors.

In [3]:
model = artifact['model']
feature_names = artifact['features']
importances = model.feature_importances_
top_idx = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(10, 6))
plt.barh([feature_names[i] for i in reversed(top_idx)], [importances[i] for i in reversed(top_idx)], color='#264653')
plt.title('Top 15 Feature Importances (Tuned XGBoost Corn Model)', fontsize=13, fontweight='bold')
plt.xlabel('Relative Feature Importance (Gain)')
plt.tight_layout()
plt.show()

Top features: W429 (July Week 29 Max Temp), W430 (July Week 30 Max Temp), Ssurface1 (Soil Productivity), S71 (Topsoil Organic Matter), W128 (Late June Rainfall).


## 3. Production Artifact Verification
The finalized model artifact encapsulates the trained XGBoost estimator, feature column list, precomputed training feature means/stds for input imputation, historical yield statistics, and performance benchmarks.

In [4]:
print("Production Artifact Payload Keys:", list(artifact.keys()))
print(f"Number of Features: {artifact['n_features']}")
print("Historical Corn Yield Bounds in Training Set:")
print(f"  Min: {artifact['stats']['yield_min']} bu/acre")
print(f"  Mean: {artifact['stats']['yield_mean']:.2f} bu/acre")
print(f"  Max: {artifact['stats']['yield_max']} bu/acre")
print("Model ready for deployment.")

Production Artifact Payload Keys: ['crop', 'model_name', 'model_version', 'best_params', 'features', 'n_features', 'model', 'metrics', 'stats', 'feature_means', 'feature_stds']
Number of Features: 400
Historical Corn Yield Bounds in Training Set:
  Min: 19.0 bu/acre
  Mean: 137.38 bu/acre
  Max: 246.7 bu/acre
Model ready for deployment.
